In [ ]:
import gymnasium as gym
import numpy as np

In [ ]:
import gymnasium as gym
env = gym.make('Blackjack-v1', render_mode="human")  
state, _ = env.reset()

In [ ]:
PLAYER_SUM_RANGE = (4, 31)  
DEALER_CARD_RANGE = (1, 10)
USABLE_ACE_RANGE = (0, 1)

class QLearningAgent:
    def __init__(self):
        self.q_table = np.zeros(
            (PLAYER_SUM_RANGE[1] + 1, 
            DEALER_CARD_RANGE[1] + 1,  
            USABLE_ACE_RANGE[1] + 1,   
            2                          
        ))
        self.alpha = 0.1  
        self.gamma = 0.9  
        self.epsilon = 0.1  

    def get_action(self, state):
        player_sum, dealer_card, usable_ace = state
        if np.random.random() < self.epsilon:
            return env.action_space.sample()  # Random action
        return np.argmax(self.q_table[player_sum, dealer_card, usable_ace])

    def update(self, state, action, reward, next_state, done):
        player_sum, dealer_card, usable_ace = state
        next_player_sum, next_dealer_card, next_usable_ace = next_state
        
        current_q = self.q_table[player_sum, dealer_card, usable_ace, action]
        next_max_q = np.max(self.q_table[next_player_sum, next_dealer_card, next_usable_ace]) if not done else 0
        
        # Q-learning update
        new_q = current_q + self.alpha * (reward + self.gamma * next_max_q - current_q)
        self.q_table[player_sum, dealer_card, usable_ace, action] = new_q

def train(episodes=100_000):
    agent = QLearningAgent()
    rewards = []
    
    for episode in range(episodes):
        observation, _ = env.reset()
        state = (
            min(max(observation[0], PLAYER_SUM_RANGE[0]), PLAYER_SUM_RANGE[1]),
            min(max(observation[1], DEALER_CARD_RANGE[0]), DEALER_CARD_RANGE[1]),
            int(observation[2])
        )
        done = False
        episode_reward = 0
        
        while not done:
            action = agent.get_action(state)
            observation, reward, done, _, _ = env.step(action)
            next_state = (
                min(max(observation[0], PLAYER_SUM_RANGE[0]), PLAYER_SUM_RANGE[1]),
                min(max(observation[1], DEALER_CARD_RANGE[0]), DEALER_CARD_RANGE[1]),
                int(observation[2])
            )
            
            agent.update(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward
        
        rewards.append(episode_reward)
        
        if episode % 10_000 == 0:
            avg_reward = np.mean(rewards[-1000:])
            print(f"Episode {episode}: Avg Reward = {avg_reward:.2f}")
    
    return agent, rewards

agent, rewards = train()

plt.figure(figsize=(10, 6))
plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid'))
plt.title("Average Reward per Episode (100-episode moving average)")
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.show()